In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -q ultralytics
!pip install -q onnxruntime-gpu --no-deps
!pip install -q fastapi uvicorn python-multipart pyngrok


import json, os
from pathlib import Path
from ultralytics import YOLO, RTDETR

model_dir = "/content/drive/MyDrive/Object-Detection-Data/models"
model_paths = {
    "yolov8m_pytorch": f"{model_dir}/yolov8m.pt",
    "yolov8m_onnx": f"{model_dir}/yolov8m.onnx",
    "yolov8m_tensorrt": f"{model_dir}/yolov8m.engine",
    "rtdetr_l_pytorch": f"{model_dir}/rtdetr-l.pt",
    "rtdetr_l_onnx": f"{model_dir}/rtdetr-l.onnx",
    "rtdetr_l_tensorrt": f"{model_dir}/rtdetr-l.engine",
}

with open("model_paths.json", "w") as f:
    json.dump(model_paths, f, indent=2)

# Verify all files exist
for name, path in model_paths.items():
    exists = "✅" if Path(path).exists() else "❌"
    print(f"  {exists} {name}: {path}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
  ✅ yolov8m_pytorch: /content/drive/MyDrive/Object-Detection-Data/models/yolov8m.pt
  ✅ yolov8m_onnx: /content/drive/MyDrive/Object-Detection-Data/models/yolov8m.onnx
  ✅ yolov8m_tensorrt: /content/drive/MyDrive/Object-Detection-Data/models/yolov8m.engine
  ✅ rtdetr_l_pytorch: /content/drive/MyDrive/Object-Detection-Data/models/rtdetr-l.pt
  ✅ rtdetr_l_onnx: /content/drive/MyDrive/Object-Detection-Data/models/rtdetr-l.onnx
  ✅ rtdetr_l_tensorrt: /content/drive/MyDrive/Object-Detection-Data/models/rtdetr-l.engine


In [3]:
%%writefile main.py
import time, json, tempfile
from pathlib import Path
import cv2, numpy as np
from fastapi import FastAPI, File, UploadFile, Query
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse
from ultralytics import YOLO, RTDETR

app = FastAPI(title="Object Detection API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_methods=["*"], allow_headers=["*"])

with open("model_paths.json") as f:
    model_paths = json.load(f)
models = {}
for key, path in model_paths.items():
    if Path(path).exists():
        print(f"Loading {key}...")
        models[key] = RTDETR(path) if "rtdetr" in key else YOLO(path)
print(f"Loaded {len(models)} models")

@app.get("/health")
def health(): return {"status": "ok", "models_loaded": list(models.keys())}

@app.get("/models")
def list_models(): return {"models": [{"key": k, "path": model_paths.get(k,"")} for k in models]}

@app.post("/detect")
async def detect(file: UploadFile = File(...), model: str = Query(default="yolov8m_pytorch"), confidence: float = Query(default=0.25, ge=0.0, le=1.0)):
    if model not in models: return JSONResponse(status_code=400, content={"error": f"Unknown model: {model}"})
    img = cv2.imdecode(np.frombuffer(await file.read(), np.uint8), cv2.IMREAD_COLOR)
    if img is None: return JSONResponse(status_code=400, content={"error": "Could not decode image"})
    start = time.perf_counter()
    results = models[model](img, conf=confidence, verbose=False)
    latency_ms = (time.perf_counter()-start)*1000
    detections = []
    for box in results[0].boxes:
        x1,y1,x2,y2 = box.xyxy[0].tolist()
        detections.append({"bbox":[round(x1,1),round(y1,1),round(x2,1),round(y2,1)], "confidence":round(float(box.conf[0]),3), "class_id":int(box.cls[0]), "class_name":results[0].names[int(box.cls[0])]})
    return {"detections": detections, "model": model, "latency_ms": round(latency_ms,1), "image_size": {"width":img.shape[1],"height":img.shape[0]}}

@app.post("/detect-video")
async def detect_video(file: UploadFile = File(...), model: str = Query(default="yolov8m_pytorch"), confidence: float = Query(default=0.25, ge=0.0, le=1.0), max_frames: int = Query(default=5000, ge=1, le=5000)):
    if model not in models: return JSONResponse(status_code=400, content={"error": f"Unknown model: {model}"})
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as tmp:
        tmp.write(await file.read()); tmp_path = tmp.name
    cap = cv2.VideoCapture(tmp_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30; total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frames_results, total_latency, frame_idx = [], 0, 0
    while cap.isOpened() and frame_idx < max_frames:
        ret, frame = cap.read()
        if not ret: break
        start = time.perf_counter()
        results = models[model](frame, conf=confidence, verbose=False)
        latency_ms = (time.perf_counter()-start)*1000; total_latency += latency_ms
        dets = []
        for box in results[0].boxes:
            x1,y1,x2,y2 = box.xyxy[0].tolist()
            dets.append({"bbox":[round(x1,1),round(y1,1),round(x2,1),round(y2,1)], "confidence":round(float(box.conf[0]),3), "class_id":int(box.cls[0]), "class_name":results[0].names[int(box.cls[0])]})
        frames_results.append({"frame":frame_idx, "detections":dets, "latency_ms":round(latency_ms,1)})
        frame_idx += 1
    cap.release(); Path(tmp_path).unlink(missing_ok=True)
    return {"frames":frames_results, "model":model, "video_fps":round(fps,1), "total_frames_processed":frame_idx, "total_frames_in_video":total_frames, "avg_latency_ms":round(total_latency/max(frame_idx,1),1)}



Writing main.py


In [4]:
NGROK_AUTH_TOKEN = "3CGi3kNPKWmjsAI5BMRFwrtmghf_7Put6ezRMDETK56oWxs6f"  # <-- token
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(8001)
print(f"Backend URL: {public_url}")

import threading, uvicorn
thread = threading.Thread(target=uvicorn.run, args=("main:app",), kwargs={"host":"0.0.0.0","port":8001})
thread.daemon = True
thread.start()

Backend URL: NgrokTunnel: "https://handprint-scalded-pretzel.ngrok-free.dev" -> "http://localhost:8001"


In [6]:
from pyngrok import ngrok
#ngrok.kill()
# Don't use the saved domain; instead gets a fresh random one
#public_url = ngrok.connect(addr="8001", bind_tls=True)
print(f"Backend URL: {public_url}")

Backend URL: NgrokTunnel: "https://handprint-scalded-pretzel.ngrok-free.dev" -> "http://localhost:8001"/health
